###Phase 8 — Explainable AI Recruiter

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pickle
import json
import pandas as pd

In [4]:
DATA_PATH="/content/drive/MyDrive/Redrob_AI_Hackathon/datasets/candidates.jsonl"

candidates=[]

with open(DATA_PATH,'r',encoding='utf-8') as f:
    for line in f:
        candidates.append(json.loads(line))

candidate_dict = {
    c["candidate_id"]: c
    for c in candidates
}

In [5]:
with open(
"/content/drive/MyDrive/Redrob_AI_Hackathon/outputs/top300_crossencoder.pkl",
"rb"
) as f:

    top300 = pickle.load(f)

In [6]:
top100 = top300[:100]

In [7]:
PRODUCT_COMPANIES = {
"Google","Amazon","Apple","Flipkart","Paytm",
"Razorpay","Zomato","CRED","Zoho","Yellow.ai",
"Netflix","Sarvam AI","Freshworks","Meesho"
}

def generate_explanation(candidate_id):

    cand = candidate_dict[candidate_id]

    scorecard_data = scorecard(candidate_id)

    profile = cand["profile"]
    sig = cand["redrob_signals"]

    strengths=[]
    risks=[]
    why=[]

    years = profile["years_of_experience"]

    if 5 <= years <= 9:
        strengths.append(
            f"{years} years experience aligns with ideal band."
        )

    current_company = profile["current_company"]

    if current_company in PRODUCT_COMPANIES:

        strengths.append(
            f"Experience in product company ({current_company})."
        )

    response = sig["recruiter_response_rate"]

    if response > 0.7:

        strengths.append(
            "Excellent recruiter response rate."
        )

    elif response > 0.4:

        strengths.append(
            "Good recruiter engagement."
        )

    notice = sig["notice_period_days"]

    if notice <= 30:

        strengths.append(
            "Short notice period."
        )

    elif notice > 90:

        risks.append(
            "Long notice period."
        )

    github = sig["github_activity_score"]

    if github > 50:

        strengths.append(
            "Strong GitHub activity."
        )

    elif github < 0:

        risks.append(
            "No public GitHub activity."
        )

    why.append(
        "High semantic relevance to search, retrieval and AI systems."
    )

    summary = (
        f"{profile['current_title']} with "
        f"{years} years experience from "
        f"{current_company}. Strong fit for AI and retrieval-focused roles."
    )

    return {

        "candidate_id":candidate_id,

        "strengths":strengths,

        "risks":risks,

        "why_selected":why,

        "summary":summary,

        "scorecard":scorecard_data
    }

###PHASE 9 — Multi-Dimensional Recruiter Scorecard

In [8]:
def scorecard(candidate_id):

    cand = candidate_dict[candidate_id]

    sig = cand["redrob_signals"]

    years = cand["profile"]["years_of_experience"]

    #################################################
    # Technical Fit
    #################################################

    technical = 70

    if 5 <= years <= 9:
        technical += 10

    github = max(sig["github_activity_score"],0)

    technical += min(github/5,20)

    technical = min(technical,100)

    #################################################
    # Behavioral Fit
    #################################################

    behavioral = (
        sig["recruiter_response_rate"]*50
        +
        sig["interview_completion_rate"]*50
    )

    #################################################
    # Availability
    #################################################

    availability = 70

    if sig["open_to_work_flag"]:
        availability += 10

    notice = sig["notice_period_days"]

    if notice <= 30:
        availability += 20

    elif notice <= 60:
        availability += 10

    availability = min(availability,100)

    #################################################
    # Risk
    #################################################

    risk = 0

    if notice > 90:
        risk += 20

    if github < 0:
        risk += 10

    if len(cand["career_history"]) >= 6:
        risk += 10

    return {

        "technical_fit":round(technical),

        "behavioral_fit":round(behavioral),

        "availability_fit":round(availability),

        "risk_score":round(risk),

        "overall_match":round(
            (
                technical*0.5
                +
                behavioral*0.3
                +
                availability*0.2
            )
        )
    }

In [9]:
explanations=[]

for cand in top100:

    explanations.append(
        generate_explanation(
            cand["candidate_id"]
        )
    )

In [10]:
for i in range(5):

    print("="*80)

    print(explanations[i])

{'candidate_id': 'CAND_0018499', 'strengths': ['7.2 years experience aligns with ideal band.', 'Experience in product company (Zomato).', 'Good recruiter engagement.', 'Short notice period.', 'Strong GitHub activity.'], 'risks': [], 'why_selected': ['High semantic relevance to search, retrieval and AI systems.'], 'summary': 'Senior Machine Learning Engineer with 7.2 years experience from Zomato. Strong fit for AI and retrieval-focused roles.', 'scorecard': {'technical_fit': 99, 'behavioral_fit': 70, 'availability_fit': 100, 'risk_score': 0, 'overall_match': 91}}
{'candidate_id': 'CAND_0061265', 'strengths': ['6.6 years experience aligns with ideal band.', 'Experience in product company (Zoho).', 'Excellent recruiter response rate.', 'Strong GitHub activity.'], 'risks': ['Long notice period.'], 'why_selected': ['High semantic relevance to search, retrieval and AI systems.'], 'summary': 'Recommendation Systems Engineer with 6.6 years experience from Zoho. Strong fit for AI and retrieval-

###PHASE 10 — Executive Recruiter Dashboard

In [11]:
def hiring_decision(score):

    if score >= 90:
        return "Strong Hire"

    elif score >= 80:
        return "Hire"

    elif score >= 70:
        return "Consider"

    else:
        return "Low Priority"

In [12]:
def confidence_level(score):

    if score >= 90:
        return "Very High"

    elif score >= 80:
        return "High"

    elif score >= 70:
        return "Moderate"

    return "Low"

In [13]:
def executive_dashboard(candidate_id):

    explanation = generate_explanation(candidate_id)

    score = explanation["scorecard"]["overall_match"]

    technical = explanation["scorecard"]["technical_fit"]

    behavioral = explanation["scorecard"]["behavioral_fit"]

    availability = explanation["scorecard"]["availability_fit"]

    risk = explanation["scorecard"]["risk_score"]

    if technical >= 90:

        interview_focus = [
            "System Design",
            "Retrieval Systems",
            "Production ML"
        ]

    elif technical >= 80:

        interview_focus = [
            "ML Fundamentals",
            "Python",
            "Feature Engineering"
        ]

    else:

        interview_focus = [
            "Problem Solving",
            "Technical Depth"
        ]

    return {

        "candidate_id":candidate_id,

        "match_percentage":score,

        "hiring_recommendation":
            hiring_decision(score),

        "confidence_level":
            confidence_level(score),

        "technical_fit":technical,

        "behavioral_fit":behavioral,

        "availability_fit":availability,

        "risk_score":risk,

        "strengths":
            explanation["strengths"],

        "risks":
            explanation["risks"],

        "executive_summary":
        explanation["summary"],

        "recommended_interview_focus":
        interview_focus
    }

In [14]:
dashboard_reports = []

for cand in top100:

    dashboard_reports.append(
        executive_dashboard(
            cand["candidate_id"]
        )
    )

In [15]:
for i in range(5):

    print("="*80)

    print(dashboard_reports[i])

    print()

{'candidate_id': 'CAND_0018499', 'match_percentage': 91, 'hiring_recommendation': 'Strong Hire', 'confidence_level': 'Very High', 'technical_fit': 99, 'behavioral_fit': 70, 'availability_fit': 100, 'risk_score': 0, 'strengths': ['7.2 years experience aligns with ideal band.', 'Experience in product company (Zomato).', 'Good recruiter engagement.', 'Short notice period.', 'Strong GitHub activity.'], 'risks': [], 'executive_summary': 'Senior Machine Learning Engineer with 7.2 years experience from Zomato. Strong fit for AI and retrieval-focused roles.', 'recommended_interview_focus': ['System Design', 'Retrieval Systems', 'Production ML']}

{'candidate_id': 'CAND_0061265', 'match_percentage': 87, 'hiring_recommendation': 'Hire', 'confidence_level': 'High', 'technical_fit': 96, 'behavioral_fit': 76, 'availability_fit': 80, 'risk_score': 20, 'strengths': ['6.6 years experience aligns with ideal band.', 'Experience in product company (Zoho).', 'Excellent recruiter response rate.', 'Strong G

###PHASE 11 — Candidate Comparison Engine

In [16]:
def compare_candidates(candidate1, candidate2):

    c1 = executive_dashboard(candidate1)
    c2 = executive_dashboard(candidate2)

    comparison = {

        "candidate_1":candidate1,
        "candidate_2":candidate2,

        "match_percentage":

        (
            c1["match_percentage"],
            c2["match_percentage"]
        ),

        "technical_fit":

        (
            c1["technical_fit"],
            c2["technical_fit"]
        ),

        "behavioral_fit":

        (
            c1["behavioral_fit"],
            c2["behavioral_fit"]
        ),

        "availability_fit":

        (
            c1["availability_fit"],
            c2["availability_fit"]
        ),

        "risk_score":

        (
            c1["risk_score"],
            c2["risk_score"]
        )

    }

    return comparison

In [17]:
comparison = compare_candidates(
    "CAND_0018499",
    "CAND_0061265"
)

comparison

{'candidate_1': 'CAND_0018499',
 'candidate_2': 'CAND_0061265',
 'match_percentage': (91, 87),
 'technical_fit': (99, 96),
 'behavioral_fit': (70, 76),
 'availability_fit': (100, 80),
 'risk_score': (0, 20)}

###PHASE 12 — JSON Dashboard Export

In [18]:
import json

In [19]:
with open(
"/content/drive/MyDrive/Redrob_AI_Hackathon/outputs/dashboard_reports.json",
"w"
) as f:

    json.dump(
        dashboard_reports,
        f,
        indent=4
    )